# EDA Behavior CERT

Exploratory analysis focused on CERT r4.2 insider threat data.

Steps:
- Inventory CERT files and LDAP logs.
- Sample LDAP records for schema and key columns.
- Run the training data audit and summarize outputs.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
import pandas as pd
from pathlib import Path

behavior_root = REPO_ROOT / 'data' / 'raw' / 'behavior'
cert_root = behavior_root / 'r4.2'
ldap_dir = cert_root / 'LDAP'

summary = {
    'cert_r4_2': {},
    'ldap': {},
}

print('CERT root:', cert_root)
if cert_root.exists():
    files = [p for p in cert_root.iterdir() if p.is_file()]
    print('CERT files:', len(files))
    for sample in files[:10]:
        print(' -', sample.name)
    summary['cert_r4_2']['file_count'] = len(files)
else:
    print('Missing CERT root:', cert_root)

print('LDAP dir:', ldap_dir)
if ldap_dir.exists():
    ldap_files = sorted([p for p in ldap_dir.rglob('*') if p.is_file()])
    summary['ldap']['file_count'] = len(ldap_files)
    print('LDAP files:', len(ldap_files))
    for sample in ldap_files[:8]:
        print(' -', sample.relative_to(REPO_ROOT))
else:
    print('Missing LDAP directory:', ldap_dir)


In [ ]:
# Sample LDAP data for schema insight.
if ldap_dir.exists():
    ldap_files = sorted([p for p in ldap_dir.rglob('*') if p.is_file()])
    sample_files = ldap_files[:2]
    ldap_samples = {}
    for path in sample_files:
        print('')
        print(f'Preview LDAP file: {path.relative_to(REPO_ROOT)}')
        try:
            df = pd.read_csv(path, nrows=50000, low_memory=False)
        except Exception:
            df = pd.read_csv(path, nrows=50000, sep='	', low_memory=False)
        ldap_samples[str(path)] = {
            'rows_sampled': int(df.shape[0]),
            'cols': int(df.shape[1]),
            'columns': list(df.columns),
        }
        print('Shape (sample):', df.shape)
        print('Columns:', list(df.columns))
        for col in ['user', 'cn', 'ou', 'department']:
            if col in df.columns:
                top_vals = df[col].value_counts().head(5).to_dict()
                ldap_samples[str(path)][f'{col}_top'] = top_vals
                print(f'Top {col}:', top_vals)
    summary['ldap']['samples'] = ldap_samples


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'eda_behavior_cert_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Generate a training data audit
run([PY, 'scripts/training_data_audit.py'])



In [ ]:
# Summarize behavior-related entries from the training data audit.
audit_path = REPO_ROOT / 'reports' / 'TRAINING_DATA.json'
if not audit_path.exists():
    print('Missing:', audit_path)
else:
    audit = json.loads(audit_path.read_text(encoding='utf-8'))
    items = [
        item for item in audit.get('required', []) + audit.get('optional', [])
        if 'behavior' in str(item.get('name', '')).lower()
    ]
    if not items:
        print('No behavior entries found in TRAINING_DATA.json')
    else:
        print('behavior datasets in audit:')
        for item in items:
            print(' -', item.get('name'), '|', item.get('status'), '|', item.get('path'))


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
